Use PCA to reduce dimensionality in high-cardinality categorical features by applying feature selection techniques (e.g., Variance Threshold, Autoencoder-based compression) before PCA to ensure optimal variance retention while minimizing computational ove

Load Task 22 dataset and inspect high-cardinality columns

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('task22_final_dataset.csv')

print("Dataset shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum().sum())

# Identify high-cardinality categorical/text columns
candidate_cols = [
    "review_title",
    "review_text",
    "text_clean",
    "text_stemmed",
    "text_lemmatized"
]

high_cardinality_summary = []

for col in candidate_cols:
    if col in df.columns:
        high_cardinality_summary.append({
            "Column": col,
            "Unique Values": df[col].nunique(dropna=True),
            "Unique Percentage": round(
                df[col].nunique(dropna=True) / len(df) * 100, 2
            )
        })

display(pd.DataFrame(high_cardinality_summary))

Dataset shape: (23483, 28)

Missing values:
0


,Column,Unique Values,Unique Percentage
0,review_title,13994,59.59
1,review_text,22635,96.39
2,text_clean,22634,96.38
3,text_stemmed,22634,96.38
4,text_lemmatized,22634,96.38


Convert review text into TF-IDF features

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [5]:
# Combine review title and review text
text_df = df[['review_title', 'review_text']].copy()
text_df = text_df.fillna("").astype(str)

combined_text = (
    text_df['review_title'] + " " + text_df['review_text']
).str.strip()

# Remove empty reviews
valid_mask = combined_text.str.len() > 0
combined_text = combined_text[valid_mask]

print("Rows with review text:", len(combined_text))

# Convert text into numerical TF-IDF features
tfidf = TfidfVectorizer(
    max_features=4000,
    min_df=2,
    max_df=0.95,
    ngram_range=(1, 2),
    stop_words='english',
    dtype=np.float32
)

tfidf_matrix = tfidf.fit_transform(combined_text)

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Features before selection:", tfidf_matrix.shape[1])

Rows with review text: 23483
TF-IDF matrix shape: (23483, 4000)
Features before selection: 4000


Apply Variance Threshold

In [6]:
from sklearn.feature_selection import VarianceThreshold

In [7]:
selector = VarianceThreshold(threshold=0.0001)

selected_matrix = selector.fit_transform(tfidf_matrix)

print("Features before selection:", tfidf_matrix.shape[1])
print("Features after selection:", selected_matrix.shape[1])
print("Features removed:", tfidf_matrix.shape[1] - selected_matrix.shape[1])

Features before selection: 4000
Features after selection: 2854
Features removed: 1146


Reduce dimensions using TruncatedSVD

In [8]:
from sklearn.decomposition import TruncatedSVD

In [32]:
# Keep at most 100 components, or fewer if features are limited
n_components = min(100, selected_matrix.shape[1] - 1)

if n_components < 1:
    raise ValueError("Not enough selected features for dimensionality reduction.")

svd = TruncatedSVD(n_components=n_components, random_state=42)

reduced_arr = svd.fit_transform(selected_matrix)

reduced_df = pd.DataFrame(reduced_arr, columns=[f'SVD_Components_{i+1}' for i in range(n_components)], index = combined_text.index)

explained_variance = svd.explained_variance_ratio_.sum()

print("Shape before SVD:", selected_matrix.shape)
print("Shape after SVD:", reduced_df.shape)
print("Total explained variance ratio:", round(explained_variance, 4))
print("Explained variance percentage:", round(explained_variance * 100, 2), "%")



Shape before SVD: (23483, 2854)
Shape after SVD: (23483, 100)
Total explained variance ratio: 0.2684
Explained variance percentage: 26.84 %


Save the reduced dataset

In [27]:
# Save reduced text features
reduced_df.to_csv('task23_svd_reduced_features.csv', index=True, index_label='original_index')

# Save dimensionality reduction summary
summary_df = pd.DataFrame({
    "Metric": [
        "Original TF-IDF features",
        "Features after VarianceThreshold",
        "SVD components",
        "Explained variance ratio",
        "Explained variance percentage"
    ],
    "Value": [
        tfidf_matrix.shape[1],
        selected_matrix.shape[1],
        reduced_df.shape[1],
        round(explained_variance, 4),
        round(explained_variance * 100, 2)
    ]
})

display(summary_df)
summary_df.to_csv('task23_dimensionality_summary.csv', index=False)

print("Task 23 outputs saved successfully.")
print("Reduced features:", reduced_df.shape)


,Metric,Value
0,Original TF-IDF features,4000.000000
1,Features after VarianceThreshold,2854.000000
2,SVD components,400.000000
3,Explained variance ratio,0.521000
4,Explained variance percentage,52.099998


Task 23 outputs saved successfully.
Reduced features: (23483, 400)


In [21]:
df.head()

,product_id,category,subcategory1,subcategory2,location,channel,customer_age,review_title,review_text,rating,...,SVD_Components_91,SVD_Components_92,SVD_Components_93,SVD_Components_94,SVD_Components_95,SVD_Components_96,SVD_Components_97,SVD_Components_98,SVD_Components_99,SVD_Components_100
0,767,Initmates,Intimate,Intimates,Mumbai,Mobile,33,Not Provided,Absolutely wonderful - silky and sexy and comfortable,4,...,0.021263,-0.007583,0.061740,-0.034241,0.052721,0.043586,0.038442,-0.049728,0.028557,0.053443
1,1080,General,Dresses,Dresses,Bangalore,Mobile,34,Not Provided,"Love this dress! it's sooo pretty. i happened to find it in a store, and i'm glad i did bc i never would have ordered it online bc it's petite. i bought a petite and am 5'8"". i love the length on me- hits just a little below the knee. would definitely be a true midi on someone who is truly petite.",5,...,0.041126,0.036823,-0.013197,-0.000849,0.036175,-0.055968,0.020157,-0.035530,-0.000255,0.002956
2,1077,General,Dresses,Dresses,Gurgaon,Mobile,60,Some major design flaws,"I had such high hopes for this dress and really wanted it to work for me. i initially ordered the petite small (my usual size) but i found this to be outrageously small. so small in fact that i could not zip it up! i reordered it in petite medium, which was just ok. overall, the top half was comfortable and fit nicely, but the bottom half had a very tight under layer and several somewhat cheap (net) over layers. imo, a major design flaw was the net over layer sewn directly into the zipper - it c",3,...,-0.007295,0.018226,-0.009503,0.013520,0.000303,0.010194,-0.000987,0.020527,0.000832,0.013711
3,1049,General Petite,Bottoms,Pants,Chennai,Web,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, flirty, and fabulous! every time i wear it, i get nothing but great compliments!",5,...,-0.049560,-0.028028,0.068227,-0.011083,-0.032856,0.039142,0.018753,-0.027390,-0.007880,-0.076504
4,847,General,Tops,Blouses,Bangalore,Web,47,Flattering shirt,This shirt is very flattering to all due to the adjustable front tie. it is the perfect length to wear with leggings and it is sleeveless so it pairs well with any cardigan. love this shirt!!!,5,...,-0.006564,0.019026,0.043036,0.014710,0.003309,-0.011144,0.000402,0.020513,0.011192,0.010737


### Conclusion – Task 23: Dimensionality Reduction Using PCA-Based Techniques

In this task, dimensionality reduction was performed on the Women’s Clothing Reviews dataset containing 23,483 records and 28 columns, with no missing values. High-cardinality text columns were identified, including `review_title`, `review_text`, `text_clean`, `text_stemmed`, and `text_lemmatized`. The review title contained 13,994 unique values, while the review text contained 22,635 unique values.

The review title and review text were combined and transformed into a TF-IDF feature matrix containing 4,000 features. The VarianceThreshold technique reduced the feature count from 4,000 to 2,854 by removing 1,146 low-variance features, representing a 28.65% reduction. Subsequently, TruncatedSVD reduced the selected features to 100 components, achieving a 96.50% reduction in dimensionality.

The 100 components retained an explained variance ratio of 26.84%. This demonstrates a substantial reduction in feature dimensions, although the retained variance indicates that additional components may be needed if greater variance retention is required. The impact of dimensionality reduction on predictive performance should also be evaluated before selecting the final number of components.

Overall, this task demonstrated how TF-IDF, feature selection, and TruncatedSVD can reduce the dimensionality of high-cardinality review text. The reduced feature dataset was saved for further analysis. Although the assignment mentions PCA and autoencoder-based compression as possible techniques, this implementation used TruncatedSVD because it is suitable for sparse TF-IDF matrices; a separate autoencoder comparison was not performed.

In [33]:
# Create a copy of the Task 22 dataset
task23_df = df.copy()

# Ensure reduced features align with the original dataset rows
if not task23_df.index.equals(reduced_df.index):
    raise ValueError(
        "Row indexes do not match. Align the reduced features first."
    )

# Add the SVD components to the original dataset
task23_df = pd.concat(
    [task23_df, reduced_df],
    axis=1
)

# Save the complete Task 23 final dataset
task23_df.to_csv(
    "task23_final_dataset.csv",
    index=False
)

print("Task 23 final dataset saved successfully!")
print("Final dataset shape:", task23_df.shape)
print("Original columns:", df.shape[1])
print("Added SVD components:", reduced_df.shape[1])
print("Final dataset: task23_final_dataset.csv")

# Verify the saved dataset
saved_df = pd.read_csv("task23_final_dataset.csv")

print("\nVerification")
print("Saved dataset shape:", saved_df.shape)
print("Missing values:", saved_df.isnull().sum().sum())

Task 23 final dataset saved successfully!
Final dataset shape: (23483, 828)
Original columns: 728
Added SVD components: 100
Final dataset: task23_final_dataset.csv

Verification
Saved dataset shape: (23483, 828)
Missing values: 0


In [23]:
saved_df.columns.tolist()

['product_id',
 'category',
 'subcategory1',
 'subcategory2',
 'location',
 'channel',
 'customer_age',
 'review_title',
 'review_text',
 'rating',
 'recommend_flag',
 'vader_negative',
 'vader_neutral',
 'vader_positive',
 'vader_compound',
 'vader_sentiment_score',
 'vader_sentiment_label',
 'age_cluster_raw',
 'age_group',
 'sentiment_target',
 'text_clean',
 'text_stemmed',
 'text_lemmatized',
 'sentiment_standard',
 'sentiment_quantile',
 'sentiment_mean_based',
 'vader_label_normalized',
 'model_text',
 'SVD_Components_1',
 'SVD_Components_2',
 'SVD_Components_3',
 'SVD_Components_4',
 'SVD_Components_5',
 'SVD_Components_6',
 'SVD_Components_7',
 'SVD_Components_8',
 'SVD_Components_9',
 'SVD_Components_10',
 'SVD_Components_11',
 'SVD_Components_12',
 'SVD_Components_13',
 'SVD_Components_14',
 'SVD_Components_15',
 'SVD_Components_16',
 'SVD_Components_17',
 'SVD_Components_18',
 'SVD_Components_19',
 'SVD_Components_20',
 'SVD_Components_21',
 'SVD_Components_22',
 'SVD_Compo

In [34]:
# Remove previously added SVD components from the original dataset
original_df = df.loc[
    :,
    ~df.columns.str.contains(r"^SVD_Components")
].copy()

# Check row alignment
if not original_df.index.equals(reduced_df.index):
    raise ValueError("Row indexes do not match.")

# Add the latest 100 SVD components only once
task23_df = pd.concat(
    [original_df, reduced_df],
    axis=1
)

# Save final dataset
task23_df.to_csv(
    "task23_final_dataset.csv",
    index=False
)

print("Original columns after removing old SVD:",
      original_df.shape[1])
print("SVD components added:", reduced_df.shape[1])
print("Final dataset shape:", task23_df.shape)

Original columns after removing old SVD: 28
SVD components added: 100
Final dataset shape: (23483, 128)


In [35]:
saved_df['SVD_Components_93'].value_counts()

SVD_Components_93
 0.001145    841
 0.020505      3
 0.008233      2
-0.039305      2
-0.020658      2
            ... 
 0.032288      1
 0.001135      1
-0.010460      1
-0.021578      1
 0.068227      1
Name: count, Length: 22634, dtype: int64

In [36]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# 1. Load dataset containing original + SVD columns
df = pd.read_csv("task23_final_dataset.csv")

# 2. Select the 100 SVD columns
svd_cols = [
    col for col in df.columns
    if col.startswith("SVD_Components_")
]

print("Dataset shape:", df.shape)
print("Number of SVD features:", len(svd_cols))

# 3. Target variable: 3-class sentiment
target_col = "rating"

if target_col not in df.columns:
    raise ValueError(
        f"{target_col} not found. Check your target column name."
    )

# 4. Prepare X and y
X = df[svd_cols].copy()
y = df[target_col].copy()

# Handle missing values, if any
X = X.replace([np.inf, -np.inf], np.nan)
valid_rows = X.notna().all(axis=1) & y.notna()
X = X.loc[valid_rows]
y = y.loc[valid_rows]

# Encode target labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y.astype(str))

print("\nTarget classes:", label_encoder.classes_)
print("\nClass distribution:")
print(y.value_counts())

# 5. Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

print("\nTraining rows:", len(X_train))
print("Testing rows:", len(X_test))

# 6. Define models
models = {
    "Logistic Regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000, class_weight="balanced")
    ),
    "Linear SVM": make_pipeline(
        StandardScaler(),
        LinearSVC(class_weight="balanced", random_state=42)
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ),
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    )
}

# 7. Train models and calculate scores
results = []
trained_models = {}

for name, model in models.items():
    print(f"\nTraining {name}...")

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    trained_models[name] = model

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision (Macro)": precision_score(
            y_test, y_pred, average="macro", zero_division=0
        ),
        "Recall (Macro)": recall_score(
            y_test, y_pred, average="macro", zero_division=0
        ),
        "F1 Score (Macro)": f1_score(
            y_test, y_pred, average="macro", zero_division=0
        )
    })

# 8. Compare scores
results_df = pd.DataFrame(results).sort_values(
    by="F1 Score (Macro)", ascending=False
)

display(results_df.round(4))

# 9. Best model report
best_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_name]
best_pred = best_model.predict(X_test)

print("\nBest model:", best_name)
print("\nClassification Report:")
print(
    classification_report(
        y_test,
        best_pred,
        labels=np.arange(len(label_encoder.classes_)),
        target_names=label_encoder.classes_,
        zero_division=0
    )
)

print("Confusion Matrix:")
display(
    pd.DataFrame(
        confusion_matrix(
            y_test,
            best_pred,
            labels=np.arange(len(label_encoder.classes_))
        ),
        index=label_encoder.classes_,
        columns=label_encoder.classes_
    )
)


Dataset shape: (23483, 128)
Number of SVD features: 100

Target classes: ['1' '2' '3' '4' '5']

Class distribution:
rating
5    13128
4     5077
3     2871
2     1565
1      842
Name: count, dtype: int64

Training rows: 18786
Testing rows: 4697

Training Logistic Regression...

Training Linear SVM...

Training Random Forest...

Training Extra Trees...


,Model,Accuracy,Precision (Macro),Recall (Macro),F1 Score (Macro)
1,Linear SVM,0.6089,0.4197,0.4559,0.4208
0,Logistic Regression,0.5625,0.4040,0.4598,0.4171
2,Random Forest,0.5785,0.5083,0.2471,0.2328
3,Extra Trees,0.5642,0.4360,0.2117,0.1673



Best model: Linear SVM

Classification Report:
              precision    recall  f1-score   support

           1       0.26      0.57      0.36       168
           2       0.27      0.28      0.28       313
           3       0.34      0.31      0.32       574
           4       0.46      0.27      0.34      1016
           5       0.77      0.85      0.81      2626

    accuracy                           0.61      4697
   macro avg       0.42      0.46      0.42      4697
weighted avg       0.60      0.61      0.60      4697

Confusion Matrix:


,1,2,3,4,5
1,96,31,25,5,11
2,100,88,84,15,26
3,84,122,180,85,103
4,37,42,156,270,511
5,52,38,92,218,2226
